### Installation

In [ ]:
%%capture

import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth

else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install --upgrade torchao

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True # 4bit quantization to reduce memory usage

fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit", # ***
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit",
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# add lora adapters

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # any number > 0. suggested: 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # 0 is optimized
    bias = "none",    # "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data Prep

In [ ]:
prompt = """Below is a tap sequence.

### Sequence:
{}

"""

EOS_TOKEN = tokenizer.eos_token # must add EOS_TOKEN so the model knows when to stop generating
def formatting_prompts_func(examples):
    sequences       = examples["sequence"]
    texts = []
    for sequence in sequences:
        text = prompt.format(sequence) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("json", data_files="training_aug.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

<a name="Train"></a>
### Training

In [ ]:
from trl import SFTConfig, SFTTrainer

val_dataset = load_dataset('json', data_files='val_aug.jsonl', split='train')
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        # max_steps = 60, # started with this, before testing full epochs
        learning_rate = 1e-4, # lowered this from an original 2e-4
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 30,
        optim = "adamw_8bit",
        weight_decay = 0.01, # raised this from 0.001
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

train_steps = []
train_loss = []
val_steps = []
val_loss = []

for log in trainer.state.log_history:
    if 'loss' in log and 'step' in log:
        train_steps.append(log['step'])
        train_loss.append(log['loss'])
    if 'eval_loss' in log and 'step' in log:
        val_steps.append(log['step'])
        val_loss.append(log['eval_loss'])

plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_loss, label='Training Loss', color='blue')

# to compare training loss to validation set
if val_steps:
    plt.plot(val_steps, val_loss, label='Validation Loss', color='red', marker='o')

plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.grid(True)
plt.legend()
plt.show()

<a name="Inference"></a>
### Inference

In [ ]:
import json
import random

FastLanguageModel.for_inference(model)

val_filename = 'val_aug.jsonl'
evaluation_results = []

with open(val_filename, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue

        data = json.loads(line)
        full_sequence = data['sequence']

        steps = full_sequence.split(' ')
        total_steps = len(steps)

        # pick a random cutoff between 2 steps and total_steps minus 2
        split_idx = random.randint(2, max(2, total_steps - 2))
        prefix = ' '.join(steps[:split_idx])

        inputs = tokenizer([prefix], return_tensors='pt').to('cuda')
        outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, pad_token_id=tokenizer.eos_token_id)

        generated_full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        # slice off the prefix to isolate ONLY what the model generated
        prefix_length = len(prefix)
        new_steps_only = generated_full_text[prefix_length:].strip()

        # store the isolated new steps for eval below
        evaluation_results.append({
            'prefix': prefix,
            'generated_output': new_steps_only,
            'actual_target': full_sequence
        })

# check out 3 model-completed sequences
for i in range(3):
    print(f" ~ VALIDATION TEST {i+1} ~ ")
    print(f"Prefix given to model : {evaluation_results[i]['prefix']}")
    print(f"Model output          : {evaluation_results[i]['generated_output']}")
    print(f"Target sequence       : {evaluation_results[i]['actual_target']}\n")

### Evaluation

In [ ]:
STEP_PHYSICS = {
    # steps that require a free foot and trap the weight
    'step': ('free', 'trapped'),
    'stamp': ('free', 'trapped'),
    'flap_heel': ('free', 'trapped'),
    'drawback': ('free', 'trapped'),
    'jump': ('free', 'trapped'),
    'ball_heel': ('free', 'trapped'),
    'heel_toe': ('free', 'trapped'),
    'paradiddle': ('free', 'trapped'),
    'pas_de_bourree': ('free', 'trapped'),
    'single_switch_pullback': ('free', 'trapped'),
    'single_switch_wing': ('free', 'trapped'),
    '4_count_riff': ('free', 'trapped'),
    '5_count_riff': ('free', 'trapped'),
    '6_count_riff': ('free', 'trapped'),
    '7_count_riff': ('free', 'trapped'),

    # steps that require a free foot and leave the foot free
    'shuffle': ('free', 'free'),
    'scuff': ('free', 'free'),
    'toe': ('free', 'free'),
    'stomp': ('free', 'free'),
    'spank': ('free', 'free'),
    'brush': ('free', 'free'),
    'dig': ('free', 'free'),
    '3_count_riff': ('free', 'free'),
    'clunk': ('free', 'free'),
    'cramp_roll': ('free', 'free'),
    'ball_change': ('free', 'free'),

    # steps that require the foot to already be bearing weight
    'heel_drop': ('bearing', 'bearing'),
    'toe_drop': ('bearing', 'bearing'),
    'hop': ('bearing', 'bearing'),
    'chug': ('bearing', 'bearing'),

    # steps that require the foot to be bearing weight, then it becomes free
    'single_pullback': ('bearing', 'free'),
    'single_wing': ('bearing', 'free'),

    # ambiguous steps (require free foot, end state ambiguous)
    'flap': ('free', 'ambiguous'),
    'heel': ('free', 'ambiguous'),

    # other ambiguous steps (unclear or open-ended)
    'pullback': ('any', 'ambiguous'),
    'double_pullback': ('any', 'ambiguous'),
    'wing': ('any', 'ambiguous'),
    'clap': ('any', 'ambiguous'),
}

In [ ]:
import re

def parse_sequence(prefix, generated_output, initial_state=None, return_state=False):
    clean_generated = generated_output.split('\n\n')[0].strip()

    # if a prefix is provided, evaluate the whole string.
    # if no prefix, JUST evaluate the generated chunk
    full_sequence = f"{prefix} {clean_generated}".strip()
    steps = full_sequence.split(' ')

    # use the initial_state if provided (for independent step grading)
    trapped_foot = initial_state # this will be None if no initial step is present (which means both feet are free)

    timeline = []
    last_full_beat = 0.0
    loop_offset = 0.0

    for i, step in enumerate(steps):
        if not step:
            continue

        # SYNTAX CHECK! are all 3 components of the token present?
        parts = step.split('|')
        if len(parts) != 3:
            error_msg = f"Missing step|foot|count format at '{step}'."
            return ("SYNTAX ERROR", error_msg, trapped_foot) if return_state else ("SYNTAX ERROR", error_msg)
        name, foot, count_str = parts[0], parts[1], parts[2]

        # PHYSICS CHECK! are the state & weight transfer valid?
        if name in STEP_PHYSICS:
            req_state, end_state = STEP_PHYSICS[name]
            if req_state == 'free':
                if trapped_foot == foot:
                    error_msg = f"Foot {foot} must be free for '{name}', but is bearing weight at '{step}'."
                    return ("PHYSICS ERROR", error_msg, trapped_foot) if return_state else ("PHYSICS ERROR", error_msg)
            elif req_state == 'bearing':
                if trapped_foot != foot and trapped_foot != 'B':
                    error_msg = f"Foot {foot} must be bearing weight for '{name}', but is free at '{step}'."
                    return ("PHYSICS ERROR", error_msg, trapped_foot) if return_state else ("PHYSICS ERROR", error_msg)
            if end_state == 'trapped':
                trapped_foot = foot
            elif end_state == 'free':
                if trapped_foot == foot:
                    trapped_foot = None
            elif end_state == 'ambiguous':
                this_foot_req = None
                other_foot = 'L' if foot == 'R' else ('R' if foot == 'L' else None)
                other_foot_req = None
                for j in range(i + 1, len(steps)):
                    n_step = steps[j]
                    if not n_step: continue
                    n_parts = n_step.split('|')
                    if len(n_parts) == 3:
                        n_req = STEP_PHYSICS.get(n_parts[0], (None, None))[0]
                        primary_n_foot = n_parts[1][0] if len(n_parts[1]) > 0 else n_parts[1]
                        if n_req in ['free', 'bearing']:
                            if primary_n_foot == foot and this_foot_req is None:
                                this_foot_req = n_req
                            if other_foot and primary_n_foot == other_foot and other_foot_req is None:
                                other_foot_req = n_req
                            if n_parts[1] == 'B':
                                if this_foot_req is None: this_foot_req = n_req
                                if other_foot_req is None: other_foot_req = n_req
                            if this_foot_req and other_foot_req: break
                if this_foot_req == 'bearing' and other_foot_req == 'bearing':
                    trapped_foot = 'B'
                elif this_foot_req == 'bearing' or other_foot_req == 'free':
                    trapped_foot = foot
                elif this_foot_req == 'free' or other_foot_req == 'bearing':
                    pass
                else:
                    trapped_foot = foot
        else:
            # VOCAB CHECK! does the step exist?
            error_msg = f"Unknown step '{name}' at '{step}'."
            return ("VOCAB ERROR", error_msg, trapped_foot) if return_state else ("VOCAB ERROR", error_msg)

        # RHYTHM CHECK! do the counts follow temporal ordering?
        for char in count_str:
            if char.isdigit():
                raw_val = float(char)
                if last_full_beat >= 7.0 and raw_val <= 2.0:
                    loop_offset += 8.0
                last_full_beat = raw_val
                val = raw_val + loop_offset
            elif char == 'e': val = last_full_beat + loop_offset + 0.25
            elif char == '&': val = last_full_beat + loop_offset + 0.5
            elif char == 'a': val = last_full_beat + loop_offset + 0.75
            else:
                error_msg = f"Invalid subdivision '{char}' at '{step}'."
                return ("RHYTHM ERROR", error_msg, trapped_foot) if return_state else ("RHYTHM ERROR", error_msg)

            if timeline and val < timeline[-1]:
                error_msg = f"Chronology is moving backwards in time at '{step}'."
                return ("RHYTHM ERROR", error_msg, trapped_foot) if return_state else ("RHYTHM ERROR", error_msg)
            timeline.append(val)

    # if we made it here: SUCCESS
    if return_state:
        return "PASSED", "Sequence is valid.", trapped_foot
    return "PASSED", "Sequence is valid."

In [ ]:
def run_evaluation(evaluation_results):
    results_summary = {
        "PASSED": 0,
        "SYNTAX ERROR": 0,
        "VOCAB ERROR": 0,
        "PHYSICS ERROR": 0,
        "RHYTHM ERROR": 0,
    }

    total = len(evaluation_results)
    if total == 0:
        print("No results to evaluate :(")
        return

    for result in evaluation_results:
        status, message = parse_sequence(result['prefix'], result['generated_output'])
        results_summary[status] += 1

        if status != "Passed":
            print(f"!! WHAT HAPPENED? {status} !!")
            print(f"  - Reason: {message}")
            print(f"  - Prefix: {result['prefix']}")
            clean_print = result['generated_output'].split('\n\n')[0].strip()
            print(f"  - Generated: {clean_print}\n")

    print("\n\n")
    print(" ~ EVALUATION SUMMARY ~ ")
    print("\n\n")
    for category, count in results_summary.items():
        percentage = (count / total) * 100
        print(f"{category:<18}: {count:<3} ({percentage:.1f}%)")

In [ ]:
run_evaluation(evaluation_results)

In [ ]:
print(" ~ STEP-LEVEL VALIDITY (let's give partial credit...) ~\n")

total_valid_steps = 0
total_generated_steps = 0
sequence_accuracies = []

for i, result in enumerate(evaluation_results):
    prefix = result['prefix']
    gen_clean = result['generated_output'].split('\n\n')[0].strip()
    gen_steps = gen_clean.split(' ') if gen_clean else []

    # first, establish the starting state from the prefix.
    # parse_sequence returns the state of the trapped_foot at the end of the prefix.
    _, _ , final_prefix_state = parse_sequence("", prefix, return_state=True)

    current_trapped_foot = final_prefix_state
    valid_count = 0

    for step in gen_steps:
        # evaluate this specific step in isolation, using the current state
        status, _ , new_state = parse_sequence("", step, initial_state=current_trapped_foot, return_state=True)

        if status == "PASSED":
            valid_count += 1
            current_trapped_foot = new_state
        else:
            # in this case, the step failed. BUT still "reset" the state to
            # whatever that step's end-state SHOULD be so grading can continue
            current_trapped_foot = new_state

    target_len = len(gen_steps)
    seq_acc = (valid_count / target_len) if target_len > 0 else 0
    sequence_accuracies.append(seq_acc)

    total_valid_steps += valid_count
    total_generated_steps += target_len

    print(f"Sequence {i+1:2d}: Valid Steps = {valid_count:2d} / {target_len:2d} | Validity Percentage = {seq_acc * 100:.1f}%")

overall_validity = sum(sequence_accuracies) / len(sequence_accuracies) if sequence_accuracies else 0
print(f"\nTotal Recoverable Accuracy: {overall_validity * 100:.1f}%")

### Unseen Sequences

In [ ]:
# similar sequences are likely to appear in training and validation sets, so try some from scratch

def generate_from_manual_prefix():
    # prompt the user for a manual string input
    prefix = input("Enter a CUSTOM tap sequence prefix in this format: step|R|1 stamp|L|2 ...:\n> ").strip()
    if not prefix:
        print("No prefix provided :(")
        return
    print("\nGenerating...")
    # tokenize the raw string, send to the GPU
    inputs = tokenizer([prefix], return_tensors='pt').to('cuda')

    # generate the completion
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # isolate the newly generated steps by slicing off the prefix
    prefix_length = len(prefix)
    new_steps_only = generated_full_text[prefix_length:].strip()
    clean_new_steps = new_steps_only.split('\n\n')[0].strip()
    print("\n\n")
    print(f"PREFIX GIVEN : {prefix}")
    print(f"MODEL OUTPUT : {clean_new_steps}")
    print("\n\n")

    # run the new output through the evaluator to verify the physics
    try:
        # establish the weight-bearing state at the exact moment your prefix ended
        _, _, initial_state = parse_sequence("", prefix, return_state=True)
        # grade the model's generated continuation
        status, message, _ = parse_sequence("", clean_new_steps, initial_state=initial_state, return_state=True)
        if status == "PASSED":
            print("PASSED: The generated sequence is valid!")
        else:
            print(f"{status}: {message}")
    except Exception as e:
        print(f"Could not run evaluation check: {e}")

for i in range(20):
  generate_from_manual_prefix()

In [ ]:
!pip freeze > requirements.txt